# O job das partidas

## Task 4: transformar a carga num job, que roda com um comando a cada Copa nova, e carregar 2026

O notebook é para explorar. Em produção, o job **roda sozinho**, chamado por um agendador, e é **versionado** como
código.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-pipeline")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
RAW_2026 = "../../data/raw_2026"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

## O script do pipeline

1. **ler** o CSV de partidas; 2. **transformar** com as funções da aula 2; 3. **gravar** em Delta por ano, com
overwrite dinâmico. As pastas chegam como parâmetros (`--raw` e `--out`).

In [ ]:
%%writefile ../../src/wc_pipeline/pipeline.py
"""Pipeline da Arena Analytics (construido na aula 5).

Le os CSVs do fornecedor, aplica as transformacoes da aula 2 e grava em Delta Lake, particionado por ano.

Uso, no terminal do Jupyter, dentro de ~/work:
    spark-submit --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py --raw data/raw --out data/out/producao
    spark-submit --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py --raw data/raw_2026 --out data/out/producao

Reexecutar e seguro: com partitionOverwriteMode=dynamic, o overwrite troca so as pastas
de ano presentes no lote e preserva as outras.
"""
import argparse

from pyspark.sql import DataFrame, SparkSession

from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao


def ler_partidas(spark: SparkSession, pasta: str) -> DataFrame:
    """A mesma funcao do video 5.1. No script, a sessao chega como parametro."""
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )


def main(pasta_raw: str, pasta_saida: str) -> None:
    spark = (
        SparkSession.builder
        .appName("arena-analytics-pipeline")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

    partidas = ler_partidas(spark, pasta_raw)
    resumo = resumo_por_selecao(partidas_por_selecao(partidas))
    anos = sorted(linha["ano"] for linha in partidas.select("ano").distinct().collect())

    for tabela, df in [("partidas", partidas), ("resumo_selecoes", resumo)]:
        (
            df.repartition("ano").write.format("delta").mode("overwrite")
            .option("partitionOverwriteMode", "dynamic")
            .partitionBy("ano")
            .save(f"{pasta_saida}/{tabela}")
        )

    total = spark.read.format("delta").load(f"{pasta_saida}/partidas").count()
    print(f"Copas carregadas neste lote: {anos}")
    print(f"Partidas em {pasta_saida}/partidas: {total}")
    spark.stop()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Pipeline da Arena Analytics")
    parser.add_argument("--raw", default="data/raw", help="pasta com os CSVs do fornecedor")
    parser.add_argument("--out", default="data/out/producao", help="pasta de saida")
    args = parser.parse_args()
    main(args.raw, args.out)

- **Sem `master`**: quem executa decide onde o job roda.
- **As configurações do Delta** (5.1) na sessão; o pacote vem pelo comando que executa o script.
- **`argparse`** lê os parâmetros da linha de comando.
- O resto já foi escrito: `ler_partidas` (5.1, com a sessão como parâmetro), as funções da aula 2 e a gravação dos vídeos 5.2 e 5.3.

## Executando com spark-submit

`spark-submit` envia um script para o Spark executar. No terminal do Jupyter, em `~/work`:

```bash
spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 \
  src/wc_pipeline/pipeline.py --raw data/raw --out data/out/producao
```

`--packages` traz o pacote do Delta, antes de o script começar.

A próxima célula já vem pronta: ela só apaga a saída de produção anterior.

In [ ]:
import shutil
shutil.rmtree(f"{OUT}/producao", ignore_errors=True)

In [ ]:
!cd ../.. && PYTHONPATH=src spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py --raw data/raw --out data/out/producao 2>/dev/null

## A Copa de 2026

Antes de carregar, a auditoria de nulos do vídeo 2.2:

In [ ]:
copa_2026 = ler_partidas(RAW_2026)

print("partidas    :", copa_2026.count())
print("público nulo:", copa_2026.filter(F.col("publico").isNull()).count())
print("data nula   :", copa_2026.filter(F.col("data_partida").isNull()).count())

Quantas partidas chegaram sem público? Um campo vazio vira `null` na leitura, e o pipeline não quebra. As agregações
ignoram os nulos (aula 2).

A carga, **duas vezes**:

In [ ]:
!cd ../.. && PYTHONPATH=src spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py --raw data/raw_2026 --out data/out/producao 2>/dev/null
!cd ../.. && PYTHONPATH=src spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py --raw data/raw_2026 --out data/out/producao 2>/dev/null

Compare o total das duas execuções: um pipeline **idempotente** dá o mesmo resultado ao rodar de novo.

## Conferindo a saída

Como no vídeo 2.3, `assert` para conferir, agora lendo a saída do pipeline:

In [ ]:
partidas_prod = spark.read.format("delta").load(f"{OUT}/producao/partidas")
resumo_prod = spark.read.format("delta").load(f"{OUT}/producao/resumo_selecoes")

total_partidas = partidas_prod.count()
total_copas = partidas_prod.select("ano").distinct().count()
gols_partidas = partidas_prod.select(F.sum("total_gols")).first()[0]
gols_selecoes = resumo_prod.select(F.sum("gols_pro")).first()[0]

assert total_partidas == 964 + 104, "o total de partidas não bate com o histórico mais 2026"
assert total_copas == 23, "falta alguma Copa na saída"
assert gols_partidas == gols_selecoes, "os gols das partidas e das seleções não batem"

print(f"saída conferida: {total_copas} Copas, {total_partidas} partidas e {gols_partidas} gols")

In [ ]:
(
    resumo_prod.filter(F.col("ano") == 2026)
    .orderBy(F.desc("vitorias"), F.desc("saldo"))
    .select("selecao", "jogos", "vitorias", "empates", "derrotas", "gols_pro", "saldo")
    .show(5)
)

## A consulta da aula 3, agora com 2026

In [ ]:
from pathlib import Path

partidas_prod.createOrReplaceTempView("partidas")
consulta = Path("../../sql/gols_por_copa.sql").read_text(encoding="utf-8")
spark.sql(consulta).filter(F.col("ano") >= 2014).show()

## Onde gravar em produção

| Lugar | Exemplo de caminho |
|---|---|
| disco local | `data/out/producao` |
| armazenamento na nuvem | `s3a://bucket/caminho` (AWS), `gs://` (Google), `abfss://` (Azure) |
| catálogo | `saveAsTable("banco.tabela")` (vídeo 3.4) |

**Nunca escreva chaves de acesso no código.**

## Do notebook para o dia a dia da empresa

| No curso | No dia a dia da empresa |
|---|---|
| `spark-submit` numa célula | um agendador (como o Airflow) roda o mesmo comando todo dia |
| `--master "local[*]"` | um cluster (`yarn`, `k8s://...`), Databricks ou EMR; o script não muda |
| `data/raw` e `data/out` | um bucket na nuvem (`s3a://`, `gs://`, `abfss://`) |
| uma Copa nova a cada 4 anos | um lote novo todo dia, regravando só a sua pasta (overwrite dinâmico) |
| conferência com `assert` | a conferência dentro do job: se falhar, o time é avisado antes do painel |
| leitura no notebook | o painel lê só a Copa que precisa; o time de analytics consulta em SQL |

Num cluster, o shuffle atravessa a rede de verdade: a aula 4 vale ainda mais. **Idempotência** e **código
versionado** são o que permite rodar isso todo dia sem susto.

## Demanda da aula 5 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | `overwrite` em Delta Lake |
| Task 2 | saída por Copa, com partition pruning |
| Task 3 | um arquivo por pasta e overwrite dinâmico |
| Task 4 | `pipeline.py` com `spark-submit`, 2026 carregada sem duplicar, saída conferida |

## Onde cada coisa foi parar

| Aula | Onde ficou | Como é usada a partir de agora |
|---|---|---|
| 1 | `src/wc_pipeline/schemas.py` | o job lê cada Copa nova com o mesmo schema |
| 2 | `src/wc_pipeline/transformacoes.py` | o job chama as três funções a cada carga |
| 3 | `sql/gols_por_copa.sql` | analytics roda a consulta sobre as partidas consolidadas |
| 4 | `data/out/passes_*`, em Parquet | estudo pontual, refeito só se pedirem |
| 5 | `src/wc_pipeline/pipeline.py` e `data/out/producao`, em Delta | roda a cada Copa nova |

Próximo passo: a atividade "Subindo o projeto no Git".

## Para consultar depois

- **Opções do `spark-submit`:** `--conf chave=valor` muda uma configuração só naquela execução, `--packages` traz
  pacotes como o do Delta, e `spark-submit --help` lista as outras.
- **Agendadores:** o Airflow, por exemplo, tem um operador que chama o `spark-submit` com os parâmetros de cada dia.
- **Conferências maiores:** em projetos grandes, as conferências com `assert` viram uma suíte de testes automatizados,
  rodada antes de cada mudança no código.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()